# Phase 2: Autoencoder Training (MLflow Engine)

---

## What This Notebook Does

This notebook is the **second phase** of our Acoustic Anomaly Detection pipeline. It takes the
scaled spectrogram matrices from Phase 1 and trains a **CNN Autoencoder** to learn the visual
"fingerprint" of normal machine sounds.

### The Core Idea: Memorize Normal, Detect Anomaly

An Autoencoder is an unsupervised neural network that learns to **copy its input to its output**
through a tight bottleneck. When trained exclusively on normal sounds:

```
Normal Sound  → Encoder → [64 numbers] → Decoder → Good Copy     (Low Error ✓)
Broken Sound  → Encoder → [64 numbers] → Decoder → Distorted Copy (High Error ✗)
```

The high reconstruction error IS our anomaly signal.

### Why Huber Loss Ignores Static Pops

Factory audio has random static pops, mic bumps, and clipping artifacts. These create
*huge* pixel errors that are NOT real anomalies. Different loss functions handle this
very differently:

| Loss | Static Pop (error=5) | Real Anomaly (error=1) | Problem |
|------|---------------------|------------------------|--------|
| **MSE** | 5² = **25** | 1² = **1** | Model wastes 96% of capacity learning noise |
| **L1** | |5| = **5** | |1| = **1** | Better, but gradients are constant (not adaptive) |
| **Huber** | L1 regime → **5** (capped) | L2 regime → **1** (precise) | **Best of both worlds** |

Huber (PyTorch's `SmoothL1Loss`) behaves like L2 for small errors (smooth, precise gradients)
but switches to L1 for large errors (robust, doesn't explode). This means our model focuses
on learning *signal*, not *noise*.

### Why AdamW Is Superior for Weight Decay

Standard Adam applies L2 regularization *inside* the adaptive learning rate calculation,
which means it gets scaled down for parameters with large gradients — partially defeating
the purpose. **AdamW** (Decoupled Weight Decay Regularization, [Loshchilov & Hutter 2019](https://arxiv.org/abs/1711.05101))
applies weight decay *directly to the weights*, independent of the gradient history.
This gives consistent, predictable regularization across all parameters.

### Why Mixup Data Augmentation

Mixup blends pairs of training spectrograms: `x_new = λ·x_a + (1-λ)·x_b` where
`λ ~ Beta(0.2, 0.2)`. This creates synthetic "in-between" samples that:
- Smooth the learned representation (reduces overfitting)
- Force the model to learn gradual transitions, not memorize exact patterns
- Are free to generate (no extra data collection needed)

---

## 1. Control Panel

Every parameter below is logged to MLflow. Change one value, re-run, and compare
results on `http://localhost:5000`.

In [6]:
TRAIN_PARAMS = {
    'LOSS_FUNCTION':  'MSE',     # 'Huber', 'L1', or 'MSE'
    'OPTIMIZER':      'AdamW',     # 'AdamW' or 'Adam'
    'BOTTLENECK_DIM': 32,          # 32, 64, or 128
    'BATCH_SIZE':     512,
    'LEARNING_RATE':  1e-3,
    'WEIGHT_DECAY':   1e-5,        # L2 reg (used by AdamW)
    'EPOCHS':         200,
    'PATIENCE':       15,          # Early stopping patience
    'PATCH_WIDTH':    32,          # Frames per patch
    'PATCH_STRIDE':   16,          # 50% overlap
    'NOISE_STD':      0.0,        # Gaussian denoising intensity
    'MIXUP_ALPHA':    0.0,         # Mixup interpolation strength
    'DROPOUT_RATE':   0.1,         # Spatial dropout rate
    'DENOISING':      False,
    'MIXUP':          False,
}

print("TRAIN CONTROL PANEL")
print("=" * 50)
for k, v in TRAIN_PARAMS.items():
    print(f"  {k:<18s} : {v}")

TRAIN CONTROL PANEL
  LOSS_FUNCTION      : MSE
  OPTIMIZER          : AdamW
  BOTTLENECK_DIM     : 32
  BATCH_SIZE         : 512
  LEARNING_RATE      : 0.001
  WEIGHT_DECAY       : 1e-05
  EPOCHS             : 200
  PATIENCE           : 15
  PATCH_WIDTH        : 32
  PATCH_STRIDE       : 16
  NOISE_STD          : 0.0
  MIXUP_ALPHA        : 0.0
  DROPOUT_RATE       : 0.1
  DENOISING          : False
  MIXUP              : False


---

## 2. Library Imports

| Library | Purpose |
|---------|--------|
| **`torch`** | The deep learning framework. Provides tensors, autograd, and the nn module. |
| **`mlflow`** | Experiment tracking — logs parameters, metrics, loss curves, and model checkpoints. |
| **`matplotlib`** | Generates loss curve plots saved as artifacts. |
| **`copy.deepcopy`** | Saves the best model state without reference aliasing. |

In [7]:
import os, json, copy, datetime
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import mlflow
import mlflow.pytorch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | PyTorch: {torch.__version__}")

Device: cuda | PyTorch: 2.5.1


---

## 3. CNN Autoencoder Architecture

### Design Decisions

| Component | Choice | Why |
|-----------|--------|-----|
| **Spatial Dropout** (`nn.Dropout2d`) | Drops entire feature maps | Adjacent pixels are highly correlated — standard dropout would have almost no effect on conv layers |
| **BatchNorm** | After every conv layer | Normalizes activations, prevents internal covariate shift, allows higher learning rates |
| **LeakyReLU(0.2)** | Instead of ReLU | Prevents "dead neurons" where gradients become zero for negative inputs |
| **True Linear Bottleneck** | `nn.Flatten → nn.Linear` | Compresses to configurable dimension (32/64/128). This is where the "fingerprint" lives. |
| **`encode()` method** | Separate method | Extracts embeddings for Isolation Forest / GMM scoring in Phase 3 |

### The Encoder-Decoder Symmetry

```
INPUT (1, H, 64) → Conv+Pool×4 → (128, H/16, 4) → Flatten → Linear → [64 numbers]
                                                                           ↓
OUTPUT (1, H, 64) ← DeConv×4 ← (128, H/16, 4) ← Reshape ← Linear ← [64 numbers]
```

In [8]:
class CNNAutoencoder(nn.Module):
    """
    CNN Autoencoder with configurable bottleneck dimension.
    Accepts patches of shape (1, input_height, PATCH_WIDTH).
    """
    def __init__(self, input_height, patch_width, bottleneck_dim, dropout_rate):
        super().__init__()
        self.input_height = input_height
        self.patch_width = patch_width

        # ENCODER: 4 conv blocks, each halves spatial dims via MaxPool2d
        self.encoder = nn.Sequential(
            self._conv_block(1, 32, dropout_rate),    # -> (32, H/2, W/2)
            self._conv_block(32, 64, dropout_rate),   # -> (64, H/4, W/4)
            self._conv_block(64, 128, dropout_rate),  # -> (128, H/8, W/8)
            self._conv_block(128, 128, dropout_rate), # -> (128, H/16, W/16)
        )

        # Calculate flattened size after 4 pooling stages
        h_enc = input_height // 16
        w_enc = patch_width // 16
        self.flat_size = 128 * h_enc * w_enc
        self.h_enc = h_enc
        self.w_enc = w_enc

        # TRUE BOTTLENECK: Flatten -> Linear compress -> Linear expand
        self.flatten = nn.Flatten()
        self.bottleneck_encode = nn.Linear(self.flat_size, bottleneck_dim)
        self.bottleneck_decode = nn.Linear(bottleneck_dim, self.flat_size)

        # DECODER: 3 deconv blocks + final layer with Sigmoid
        self.decoder = nn.Sequential(
            self._deconv_block(128, 128),
            self._deconv_block(128, 64),
            self._deconv_block(64, 32),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def _conv_block(self, in_c, out_c, dropout_rate):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, 1, 1), nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2), nn.MaxPool2d(2, 2), nn.Dropout2d(dropout_rate))

    def _deconv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, 3, 2, 1, output_padding=1),
            nn.BatchNorm2d(out_c), nn.LeakyReLU(0.2))

    def forward(self, x):
        enc = self.encoder(x)
        z = self.bottleneck_encode(self.flatten(enc))
        exp = self.bottleneck_decode(z).view(-1, 128, self.h_enc, self.w_enc)
        return self.decoder(exp)

    def encode(self, x):
        """Extract bottleneck embeddings (for Isolation Forest / GMM)."""
        return self.bottleneck_encode(self.flatten(self.encoder(x)))

print("CNNAutoencoder class defined.")

CNNAutoencoder class defined.


---

## 4. Helper Functions

In [9]:
def extract_patches(data, width, stride):
    """Slice (N, H, T) spectrograms into (M, H, width) patches."""
    patches = []
    for spec in data:
        for s in range(0, spec.shape[1] - width + 1, stride):
            patches.append(spec[:, s:s + width])
    return np.array(patches)

def get_loss_fn(name):
    """Return the loss function matching the Control Panel string."""
    if name == 'Huber': return nn.SmoothL1Loss()  # Huber = SmoothL1 in PyTorch
    elif name == 'L1':  return nn.L1Loss()
    else:               return nn.MSELoss()

def get_optimizer(name, params, lr, wd):
    """Return the optimizer matching the Control Panel string."""
    if name == 'AdamW': return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    else:               return torch.optim.Adam(params, lr=lr)

def mixup_batch(batch, alpha=0.2):
    """Mixup: blend random pairs within the batch."""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(batch.size(0)).to(batch.device)
    return lam * batch + (1 - lam) * batch[idx]

print("Helpers defined.")

Helpers defined.


---

## 5. Training Loop (with MLflow Logging)

### The Dimension Crop Fix (513 → 512)

With N_FFT=1024, Linear spectrograms have height = `(1024/2)+1` = **513**.
Our encoder has 4 `MaxPool2d(2,2)` layers that each halve the height, and the
decoder has 4 `ConvTranspose2d(stride=2)` that each double it.

513 is **not divisible by 16** (513/16 = 32.0625), so the decoder would
reconstruct a shape that doesn't match the input → **crash**.

**Fix:** Crop the height to `(H // 16) * 16` = **512** before training.
We lose 1 frequency bin out of 513 — completely negligible.

### What Gets Logged to MLflow

| What | How |
|------|-----|
| All hyperparameters | `mlflow.log_params()` |
| Per-epoch train/val loss | `mlflow.log_metrics()` |
| Best validation loss | `mlflow.log_metric()` |
| Loss curve image | `mlflow.log_artifact()` |
| Best model checkpoint | `mlflow.log_artifact()` |

In [10]:
# --- Local paths (self-contained briefcase) ---
PROC_BASE  = os.path.join(".", "data", "processed")
MODELS_DIR = os.path.join(".", "experiments", "models")
RUNS_DIR   = os.path.join(".", "experiments", "runs")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

timestamp = datetime.datetime.now().strftime("run_%Y%m%d_%H%M%S")
run_folder = os.path.join(RUNS_DIR, timestamp)
os.makedirs(run_folder, exist_ok=True)

with open(os.path.join(run_folder, "train_params.json"), 'w') as f:
    json.dump(TRAIN_PARAMS, f, indent=2)
print(f"Run folder: {run_folder}")

machines = sorted([d for d in os.listdir(PROC_BASE)
                   if os.path.isdir(os.path.join(PROC_BASE, d))])

PW = TRAIN_PARAMS['PATCH_WIDTH']
PS = TRAIN_PARAMS['PATCH_STRIDE']
BS = TRAIN_PARAMS['BATCH_SIZE']

mlflow.set_experiment("AcousticAnomalyDetection")

for machine in machines:
    print(f"\n{'='*70}")
    print(f"TRAINING: {machine}")
    print(f"{'='*70}")

    tr_path = os.path.join(PROC_BASE, machine, "X_train.npy")
    va_path = os.path.join(PROC_BASE, machine, "X_val.npy")
    if not os.path.exists(tr_path):
        print("  Missing data, skipping."); continue

    X_train_full = np.load(tr_path)
    X_val_full   = np.load(va_path)

    meta_path = os.path.join(PROC_BASE, machine, "prep_meta.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f: prep_meta = json.load(f)
        input_height = prep_meta['SPEC_HEIGHT']
    else:
        input_height = X_train_full.shape[1]

    # Extract patches
    X_tr_patches = extract_patches(X_train_full, PW, PS)
    X_va_patches = extract_patches(X_val_full, PW, PS)

    # Convert to tensors: (B, 1, H, PW)
    X_tr_t = torch.tensor(X_tr_patches, dtype=torch.float32).unsqueeze(1).to(device)
    X_va_t = torch.tensor(X_va_patches, dtype=torch.float32).unsqueeze(1).to(device)

    # ===== THE DIMENSION CROP FIX =====
    # Height must be divisible by 16 (4 pooling layers, each /2)
    # Linear spectrograms with N_FFT=1024 have H=513 → crop to 512
    H_patch = X_tr_t.shape[2]
    H_crop = (H_patch // 16) * 16
    if H_crop != H_patch:
        print(f"  [CROP FIX] Height {H_patch} → {H_crop} (must be ÷16)")
        X_tr_t = X_tr_t[:, :, :H_crop, :]
        X_va_t = X_va_t[:, :, :H_crop, :]
    input_height = H_crop  # Dynamically update for model constructor
    # ==================================

    train_loader = DataLoader(TensorDataset(X_tr_t, X_tr_t), batch_size=BS, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_va_t, X_va_t), batch_size=BS, shuffle=False)

    model = CNNAutoencoder(
        input_height, PW, TRAIN_PARAMS['BOTTLENECK_DIM'],
        TRAIN_PARAMS['DROPOUT_RATE']).to(device)
    criterion = get_loss_fn(TRAIN_PARAMS['LOSS_FUNCTION'])
    optimizer = get_optimizer(
        TRAIN_PARAMS['OPTIMIZER'], model.parameters(),
        TRAIN_PARAMS['LEARNING_RATE'], TRAIN_PARAMS['WEIGHT_DECAY'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)

    # ---- MLflow Run ----
    with mlflow.start_run(run_name=f"train_{machine}_{timestamp}"):
        mlflow.log_params({f"train_{k}": v for k, v in TRAIN_PARAMS.items()})
        mlflow.log_param("machine", machine)
        mlflow.log_param("input_height", input_height)

        best_val, patience_ctr, best_state = float('inf'), 0, None
        train_losses, val_losses = [], []

        for epoch in range(1, TRAIN_PARAMS['EPOCHS'] + 1):
            model.train()
            epoch_loss = 0
            for batch, _ in train_loader:
                batch = batch.to(device)
                if TRAIN_PARAMS['MIXUP']:
                    batch = mixup_batch(batch, TRAIN_PARAMS['MIXUP_ALPHA'])
                if TRAIN_PARAMS['DENOISING']:
                    noisy = (batch + TRAIN_PARAMS['NOISE_STD'] * torch.randn_like(batch)).clamp(0, 1)
                else:
                    noisy = batch
                optimizer.zero_grad()
                loss = criterion(model(noisy), batch)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            train_loss = epoch_loss / len(train_loader)
            train_losses.append(train_loss)

            model.eval()
            v_loss = 0
            with torch.no_grad():
                for batch, _ in val_loader:
                    batch = batch.to(device)
                    v_loss += criterion(model(batch), batch).item()
            val_loss = v_loss / len(val_loader)
            val_losses.append(val_loss)

            scheduler.step(val_loss)
            mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss}, step=epoch)

            if val_loss < best_val:
                best_val = val_loss
                best_state = copy.deepcopy(model.state_dict())
                patience_ctr = 0
                tag = "*BEST*"
            else:
                patience_ctr += 1
                tag = f"{patience_ctr}/{TRAIN_PARAMS['PATIENCE']}"

            # Print EVERY epoch
            if epoch % 1 == 0:
                print(f"  Ep {epoch:>3} | Tr: {train_loss:.6f} | Va: {val_loss:.6f} | {tag}")

            if patience_ctr >= TRAIN_PARAMS['PATIENCE']:
                print(f"  ⏹ Early stop at epoch {epoch}"); break

        # Save best model
        model_path = os.path.join(MODELS_DIR, f"best_model_{machine}.pth")
        torch.save(best_state, model_path)
        mlflow.log_artifact(model_path)
        mlflow.log_metric("best_val_loss", best_val)

        # Save loss curves
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(train_losses, label='Train')
        ax.plot(val_losses, label='Validation')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title(f'{machine} — Train vs Val Loss')
        ax.legend(); ax.grid(True, alpha=0.3)
        curve_path = os.path.join(run_folder, f"loss_curve_{machine}.png")
        fig.savefig(curve_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        mlflow.log_artifact(curve_path)

        print(f"  ✓ Best val: {best_val:.6f} | Model saved.")

print(f"\n{'='*70}")
print(f"ALL TRAINING COMPLETE")
print(f"Run folder: {run_folder}")
print(f"{'='*70}")

Run folder: .\experiments\runs\run_20260424_001819

TRAINING: ToyCar
  Ep   1 | Tr: 0.024661 | Va: 0.018285 | *BEST*
  Ep   2 | Tr: 0.015976 | Va: 0.014779 | *BEST*
  Ep   3 | Tr: 0.014675 | Va: 0.013907 | *BEST*
  Ep   4 | Tr: 0.013734 | Va: 0.013200 | *BEST*
  Ep   5 | Tr: 0.013131 | Va: 0.012727 | *BEST*
  Ep   6 | Tr: 0.012726 | Va: 0.012402 | *BEST*
  Ep   7 | Tr: 0.012403 | Va: 0.012117 | *BEST*
  Ep   8 | Tr: 0.012192 | Va: 0.012069 | *BEST*
  Ep   9 | Tr: 0.012025 | Va: 0.011869 | *BEST*
  Ep  10 | Tr: 0.011918 | Va: 0.011863 | *BEST*
  Ep  11 | Tr: 0.011784 | Va: 0.011606 | *BEST*
  Ep  12 | Tr: 0.011641 | Va: 0.011468 | *BEST*
  Ep  13 | Tr: 0.011539 | Va: 0.011367 | *BEST*
  Ep  14 | Tr: 0.011437 | Va: 0.011295 | *BEST*
  Ep  15 | Tr: 0.011331 | Va: 0.011227 | *BEST*
  Ep  16 | Tr: 0.011269 | Va: 0.011136 | *BEST*
  Ep  17 | Tr: 0.011176 | Va: 0.011021 | *BEST*
  Ep  18 | Tr: 0.011095 | Va: 0.011028 | 1/15
  Ep  19 | Tr: 0.011035 | Va: 0.010899 | *BEST*
  Ep  20 | Tr: 0.0109